In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
import os
import urllib.request
import zipfile
from collections import Counter

In [2]:
url = "http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"
urllib.request.urlretrieve(url, "cornell.zip")

with zipfile.ZipFile("cornell.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [3]:
lines = {}
conversations = []

with open("cornell movie-dialogs corpus/movie_lines.txt", encoding='iso-8859-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")
        lines[parts[0]] = parts[-1].strip()

with open("cornell movie-dialogs corpus/movie_conversations.txt", encoding='iso-8859-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")
        ids = eval(parts[-1])
        conversations.append(ids)

In [4]:
pairs = []

for conv in conversations:
    for i in range(len(conv)-1):
        input_line = lines[conv[i]]
        target_line = lines[conv[i+1]]
        pairs.append((input_line, target_line))

print(len(pairs))

221616


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text)
    return text.strip()

pairs = [(clean_text(q), clean_text(a)) for q, a in pairs]

pairs = pairs[:10000]

In [6]:
all_words = []

for q, a in pairs:
    all_words.extend(q.split())
    all_words.extend(a.split())

counter = Counter(all_words)

vocab = {word: i+4 for i, (word, _) in enumerate(counter.items())}
vocab["<pad>"] = 0
vocab["<unk>"] = 1
vocab["<sos>"] = 2
vocab["<eos>"] = 3

inv_vocab = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

In [7]:
def encode(sentence):
    return [vocab["<sos>"]] + \
           [vocab.get(word, vocab["<unk>"]) for word in sentence.split()] + \
           [vocab["<eos>"]]

data = [(encode(q), encode(a)) for q, a in pairs]

In [8]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 15

def pad(seq):
    return seq[:MAX_LEN] + [0]*(MAX_LEN - len(seq[:MAX_LEN]))

class ChatDataset(Dataset):
    def __init__(self, pairs):
        self.data = pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        q, a = self.data[idx]
        return torch.tensor(pad(q)), torch.tensor(pad(a))

dataset = ChatDataset(data)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embed = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embed)
        return outputs, hidden, cell

In [10]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size*2, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]
        hidden = hidden.repeat(seq_len, 1, 1).permute(1,0,2)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

In [11]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        embed = self.embedding(x)

        attn_weights = self.attention(hidden[-1], encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)

        context = torch.bmm(attn_weights, encoder_outputs)

        lstm_input = torch.cat((embed, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

In [12]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        vocab_size = len(vocab)

        outputs = torch.zeros(batch_size, trg_len, vocab_size)

        encoder_outputs, hidden, cell = self.encoder(src)

        x = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(x, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            x = trg[:, t]

        return outputs

In [13]:
encoder = Encoder(vocab_size, 128, 256)
decoder = Decoder(vocab_size, 128, 256)

model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
EPOCHS = 10

for epoch in range(EPOCHS):
    total_loss = 0

    for X, Y in loader:
        optimizer.zero_grad()

        output = model(X, Y)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        target = Y[:, 1:].reshape(-1)

        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss/len(loader)}")

Epoch 0, Loss: 6.061041988884679
Epoch 1, Loss: 5.348788051178661
Epoch 2, Loss: 5.072689786124915
Epoch 3, Loss: 4.85692079684224
Epoch 4, Loss: 4.6640753974548925
Epoch 5, Loss: 4.476731391760488
Epoch 6, Loss: 4.290669591282122
Epoch 7, Loss: 4.091029126804096
Epoch 8, Loss: 3.8821106733986364
Epoch 9, Loss: 3.6620573251011272


In [15]:
def predict(sentence, max_len=15):
    model.eval()

    sentence = clean_text(sentence)
    seq = torch.tensor([pad(encode(sentence))])

    encoder_outputs, hidden, cell = model.encoder(seq)

    x = torch.tensor([vocab["<sos>"]])

    result = []

    for _ in range(max_len):
        output, hidden, cell = model.decoder(x, hidden, cell, encoder_outputs)
        predicted = output.argmax(1).item()

        if predicted == vocab["<eos>"]:
            break

        result.append(inv_vocab.get(predicted, ""))
        x = torch.tensor([predicted])

    return " ".join(result)

In [16]:
while True:
    inp = input("You: ")
    if inp == "exit":
        break
    print("Bot:", predict(inp))

You:  Hi


Bot: what


You:  Hi How are you


Bot: i m not gonna be a little


You:  what?


Bot: you re a little ice


You:  what time is it 


Bot: it s a good time to be a lot of a bitch


You:  exit
